# Data Drift monitoring with EvidentlyThis notebook reproduces what `fraud_detection/components/data_validation.py` runsinside the training pipeline.

In [ ]:
import numpy as np
import pandas as pd
from evidently import Report
from evidently.presets import DataDriftPreset

pd.set_option("display.max_columns", None)

In [ ]:
df = pd.read_csv("../data/fraud_transactions.csv")
print(df.shape)
df.head()

## Reference and current windows

The reference window is the first half of the simulated month, the current window is thesecond half. In the training pipeline the two windows are the train and the test split.

In [ ]:
reference = df[df["step"] <= df["step"].median()].drop(columns=["isFraud"])
current = df[df["step"] > df["step"].median()].drop(columns=["isFraud"])

print("reference:", reference.shape)
print("current  :", current.shape)

In [ ]:
report = Report(metrics=[DataDriftPreset()])
snapshot = report.run(current_data=current, reference_data=reference)
snapshot

## Extracting the drift metrics programmatically

In [ ]:
report_dict = snapshot.dict()

rows = []
for metric in report_dict["metrics"]:
    config = metric.get("config", {})
    value = metric.get("value")

    if config.get("type", "").endswith("DriftedColumnsCount"):
        print("drifted columns:", value["count"], " share:", round(value["share"], 3))
    elif config.get("type", "").endswith("ValueDrift"):
        threshold = float(config.get("threshold", 0.05))
        method = str(config.get("method", ""))
        drifted = value < threshold if "p_value" in method else value > threshold
        rows.append(
            {"column": config["column"], "method": method,
             "score": value, "threshold": threshold, "drift_detected": drifted}
        )

pd.DataFrame(rows).sort_values("score")

## Simulating drift

Shifting the amount distribution of the current window should be picked up immediately.

In [ ]:
drifted_current = current.copy()
drifted_current["amount"] = drifted_current["amount"] * 3.0 + 50_000

drift_snapshot = Report(metrics=[DataDriftPreset()]).run(
    current_data=drifted_current, reference_data=reference
)

for metric in drift_snapshot.dict()["metrics"]:
    config = metric.get("config", {})
    if config.get("column") == "amount":
        print(config["type"], config["method"], "->", metric["value"])
    if config.get("type", "").endswith("DriftedColumnsCount"):
        print("drifted columns:", metric["value"])

## Saving the report

In [ ]:
drift_snapshot.save_html("drift_report.html")
print("saved drift_report.html")